# 🔥 Never-AI — Super-Trainer GPU « From Scratch » (v2.0.1 — Sans Blocage Google Drive)

Ce notebook entraîne **votre propre architecture neuronale `ChronoMorph-LM` (`Sprout-MoE` — ~72M paramètres, 8 experts MoE, mémoire Titans, optimiseur Muon)** sur un vrai GPU **NVIDIA T4 / L4 / A100** gratuit de Google Colab.

✅ **Corrigé v2.0.1 (`credential propagation`)** : fonctionne **100 % sans Google Drive** (compatible comptes supervisés / Family Link / cookies bloqués sur Chrome) et télécharge directement votre cerveau **`never-ai-brain-sprout-moe.zip`** prêt à importer en 1 clic dans Never-AI Studio !

### Mode d'emploi en 3 étapes :
1. Menu **Exécution** (`Runtime`) → **Modifier le type d'exécution** → sélectionnez **T4 GPU** → **Enregistrer**.
2. Cliquez sur **Exécution → Tout exécuter** (`Ctrl+F9`).
3. À la fin, Colab télécharge **`never-ai-brain-sprout-moe.zip`** : ouvrez Never-AI → bouton **⚡ Super-Cortex** → **📥 Importer `.zip` Colab** !

In [ ]:
# 1️⃣ VÉRIFICATION GPU & DOSSIER DE SAUVEGARDE (SANS ERREUR GOOGLE DRIVE !)
import os
import torch

# Par défaut : stockage direct sur le SSD ultra-rapide de Colab (/content/NeverAI_Vault)
# -> Évite 100 % des erreurs 'MessageError: credential propagation was unsuccessful'
#    causées par Family Link / contrôle parental ou le blocage des cookies tiers de Chrome.
TRY_GOOGLE_DRIVE = False  # Passez à True uniquement si vous souhaitez aussi copier sur Google Drive

VAULT_DIR = '/content/NeverAI_Vault/checkpoints'
DRIVE_BACKUP_DIR = None

if TRY_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_BACKUP_DIR = '/content/drive/MyDrive/NeverAI_Vault/checkpoints'
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print('☁️ Google Drive connecté en bonus :', DRIVE_BACKUP_DIR)
    except Exception as err:
        print(f'ℹ️ Google Drive ignoré ({err}) -> utilisation du SSD local Colab + export .zip direct !')

os.makedirs(VAULT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_label = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (Activez T4 GPU dans Exécution -> Modifier le type d\'exécution !)'
print(f'✅ Accélérateur actif : {gpu_label} | PyTorch {torch.__version__}')
print(f'✅ Coffre neuronal prêt : {VAULT_DIR}')

In [ ]:
# 2️⃣ CLONAGE NEVER-AI & INSTALLATION ULTRA-RAPIDE (~8 SECONDES SUR PYTHON 3.13)
import os, subprocess, sys

if not os.path.exists('/content/never-ai/.git'):
    subprocess.run([
        'git', 'clone', '-b', 'arena/403f5d7f-never-ai', '--depth', '1',
        'https://github.com/mouusieur/never-ai.git', '/content/never-ai'
    ], check=True)
else:
    subprocess.run(['git', '-C', '/content/never-ai', 'pull', '--ff-only'], check=False)

os.chdir('/content/never-ai')
# On conserve le PyTorch CUDA préinstallé de Colab (--no-deps) pour aller 10× plus vite !
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers', 'safetensors', 'sympy', 'pyyaml', 'tqdm', 'datasets'
], check=True)

if '/content/never-ai/src' not in sys.path:
    sys.path.insert(0, '/content/never-ai/src')
print('✅ Moteur Never-AI installé et prêt sur', os.getcwd())

In [ ]:
# 3️⃣ TÉLÉCHARGEMENT EN STREAMING DE VRAIS DIALOGUES FRANÇAIS + WIKIPEDIA FR
from datasets import load_dataset
from never_ai.core.special_tokens import BOS_TOKEN, EOS_TOKEN, NEVER_TOKEN, USER_TOKEN
from never_ai.data.curriculum import synthesize_bilingual_curriculum

N_WIKI_ARTICLES = 6000   # Montez à 25000+ pour une session longue (30-45 min sur T4)
N_DIALOGUE_PAIRS = 4000  # Vrais échanges question/réponse humains en français

raw_texts_for_tokenizer = []
formatted_docs = []

# A) Vrais dialogues conversationnels francophones (OpenAssistant OASST2)
print('💬 Téléchargement des dialogues naturels en français (OpenAssistant OASST2)...')
try:
    ds_oasst = load_dataset('OpenAssistant/oasst2', split='train', streaming=True)
    by_id = {}
    pairs = 0
    for row in ds_oasst:
        if row.get('lang') != 'fr':
            continue
        msg_id = row.get('message_id')
        parent_id = row.get('parent_id')
        role = row.get('role')
        txt = (row.get('text') or '').strip()
        if not txt:
            continue
        by_id[msg_id] = (role, txt, parent_id)
        if role == 'assistant' and parent_id in by_id and by_id[parent_id][0] == 'prompter':
            q_txt = by_id[parent_id][1]
            raw_texts_for_tokenizer.append(q_txt + '\n' + txt)
            formatted_docs.append(f'{BOS_TOKEN}{USER_TOKEN}{q_txt}{NEVER_TOKEN}{txt}{EOS_TOKEN}')
            pairs += 1
            if pairs >= N_DIALOGUE_PAIRS:
                break
    print(f'   -> {pairs} vrais dialogues français chargés !')
except Exception as e:
    print(f'   -> OASST2 indisponible ({e}), passage direct à Wikipedia FR + Curriculum.')

# B) Encyclopédie Wikipedia FR (culture générale, sciences, histoire, grammaire fluide)
print(f'📚 Streaming de {N_WIKI_ARTICLES} articles riches de Wikipedia FR...')
ds_wiki = load_dataset('wikimedia/wikipedia', '20231101.fr', split='train', streaming=True)
wiki_count = 0
for row in ds_wiki:
    txt = (row.get('text') or '').strip()
    title = (row.get('title') or '').strip()
    if len(txt) > 500:
        excerpt = txt[:3500]
        raw_texts_for_tokenizer.append(excerpt)
        # 50% en article pur, 50% en format Question -> Explication fluide !
        if wiki_count % 2 == 0 and title:
            intro = excerpt.split('\n\n')[0].strip()
            if len(intro) > 120:
                formatted_docs.append(
                    f'{BOS_TOKEN}{USER_TOKEN}Explique-moi clairement ce qu\'est {title}.{NEVER_TOKEN}{excerpt[:1800]}{EOS_TOKEN}'
                )
            else:
                formatted_docs.append(f'{BOS_TOKEN}{excerpt}{EOS_TOKEN}')
        else:
            formatted_docs.append(f'{BOS_TOKEN}{excerpt}{EOS_TOKEN}')
        wiki_count += 1
        if wiki_count >= N_WIKI_ARTICLES:
            break

# C) Curriculum souverain Never-AI (outils, code Python, raisonnement mathématique)
cur_train, cur_val = synthesize_bilingual_curriculum(seed=42, scale_factor=4)
formatted_docs.extend(cur_train)
raw_texts_for_tokenizer.extend(cur_train[:300])

total_chars = sum(len(t) for t in formatted_docs)
print(f'✅ Corpus assemblé : {len(formatted_docs):,} documents ({total_chars / 1e6:.1f} Mo de français naturel & raisonnement)')

In [ ]:
# 4️⃣ ENTRAÎNEMENT GPU INTENSIF : CHRONOMORPH SPROUT-MOE (~72M PARAMS + MUON + WSM)
import random, shutil, time
import torch
from never_ai.core.config import ChronoMorphConfig
from never_ai.core.model import ChronoMorphLM
from never_ai.core.tokenizer import MorphTokenizer
from never_ai.data.curriculum import ContinualDatasetAndReplay
from never_ai.optim.composite_muon import CompositeMuonOptimizer, WSMCheckpointMerger
from never_ai.vault.manager import VaultManager

# Réglages GPU (ajustez STEPS selon le temps souhaité : 1200 steps ≈ 5-7 min sur T4 GPU ; 5000 steps ≈ 22 min)
STEPS = 1200
BATCH_SIZE = 12 if device == 'cuda' else 2
SEQ_LEN = 384 if device == 'cuda' else 128

print('🔤 Entraînement du tokenizer BPE Byte-Level (vocab=8192) sur le français naturel...')
tok = MorphTokenizer.train_from_iterator(raw_texts_for_tokenizer, vocab_size=8192)

cfg = ChronoMorphConfig.sprout_moe(vocab_size=tok.vocab_size)
model = ChronoMorphLM(cfg).to(device)
counts = model.count_parameters()
print(f"🧠 Modèle {cfg.stage_name} sur {device.upper()} : {counts['total_params']/1e6:.2f}M paramètres totaux ({counts['active_params']/1e6:.2f}M actifs/token)")

rng = random.Random(42)
rng.shuffle(formatted_docs)
split_idx = max(16, int(len(formatted_docs) * 0.97))
dataset = ContinualDatasetAndReplay(tok, seq_len=SEQ_LEN)
stats_ds = dataset.load_corpora(formatted_docs[:split_idx], formatted_docs[split_idx:] + cur_val)
print(f"📦 Séquences empaquetées (longueur {SEQ_LEN}) : {stats_ds['train_chunks']:,} train | {stats_ds['val_chunks']:,} val")

opt = CompositeMuonOptimizer(model, muon_lr=3.0e-3, adamw_lr=8.0e-4, weight_decay=0.01, warmup_steps=80)
wsm = WSMCheckpointMerger(max_snapshots=6)
vault = VaultManager(vault_root=VAULT_DIR)
amp_dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32

t0 = time.time()
tokens_seen = 0
model.train()
for step in range(1, STEPS + 1):
    batch = dataset.sample_batch(batch_size=BATCH_SIZE, split='train', replay_ratio=0.20, device=device)
    opt.zero_grad()
    with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(amp_dtype != torch.float32)):
        out = model(batch.input_ids, targets=batch.targets, loss_mask=batch.loss_mask, update_titans=True)
        loss = out.loss
    loss.backward()
    opt.step(model)
    model.update_moe_biases()
    tokens_seen += int(batch.input_ids.numel())

    if step % 100 == 0 or step == 1:
        dt = max(1e-3, time.time() - t0)
        tok_s = tokens_seen / dt
        print(f'⚡ Step {step:4d}/{STEPS} | CE Loss = {out.ce_loss:.4f} | Vitesse = {tok_s:,.0f} tok/s | Tokens = {tokens_seen:,}')

    if step % 200 == 0 or step == STEPS:
        wsm.record_snapshot(model)
        consolidated = wsm.build_consolidated_state_dict(fallback_model=model)
        ckpt_path = vault.save_checkpoint(
            'latest',
            model,
            tok,
            metrics={'val_ce_loss': round(float(out.ce_loss), 4), 'step': step, 'tokens_trained': tokens_seen},
            consolidated_state_dict=consolidated,
        )
        if DRIVE_BACKUP_DIR:
            shutil.copytree(ckpt_path, os.path.join(DRIVE_BACKUP_DIR, 'latest'), dirs_exist_ok=True)
        print(f'💾 Checkpoint WSM sauvegardé -> {ckpt_path}')

In [ ]:
# 5️⃣ TEST EN DIRECT DE VOTRE NOUVEAU CERVEAU + TÉLÉCHARGEMENT .ZIP POUR NEVER-AI !
import os, zipfile
from never_ai.core.special_tokens import BOS_TOKEN, EOS_ID, NEVER_TOKEN, USER_ID, USER_TOKEN

model.eval()
test_questions = [
    'Salut ! Qui es-tu et comment fonctionne la gravité ?',
    'Explique-moi clairement ce qu\'est la France.',
]
stop_ids = {EOS_ID, USER_ID}

print('🧪 Aperçu direct de génération par votre ChronoMorph-LM entraîné :\n' + '-' * 65)
for q in test_questions:
    p_ids = tok.encode(f'{BOS_TOKEN}{USER_TOKEN}{q}{NEVER_TOKEN}')
    gen_ids, _ = model.generate(p_ids, max_new_tokens=90, temperature=0.65, top_p=0.9, stop_token_ids=stop_ids)
    reply = tok.decode(gen_ids[len(p_ids):], skip_special_tokens=True).strip()
    print(f'👤 Vous     : {q}')
    print(f'🤖 Never-AI : {reply}\n')

# Empaquetage en archive .zip prête pour l'import 1-clic dans Never-AI Studio (et GitHub Releases)
zip_out = '/content/never-ai-brain-sprout-moe.zip'
latest_dir = os.path.join(VAULT_DIR, 'latest')
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(latest_dir)):
        fpath = os.path.join(latest_dir, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)

size_mb = os.path.getsize(zip_out) / (1024 * 1024)
print(f'📦 Archive prête : {zip_out} ({size_mb:.1f} Mo)')
print('👉 Dans Never-AI Studio : cliquez sur ⚡ Super-Cortex -> 📥 Importer .zip Colab !')

try:
    from google.colab import files
    files.download(zip_out)
except Exception as e:
    print(f'ℹ️ Téléchargement manuel disponible dans le panneau Fichiers (icône 📁 à gauche) : {zip_out} ({e})')